In [21]:
# ==========================
# Python Libraries
# ==========================

In [22]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

In [23]:
# ==========================
# Configuration Classes
# ==========================

In [24]:
@dataclass
class EnvironmentConfig:
    grid_size: int = 5


@dataclass
class RewardConfig:
    step_reward: int = -1
    invalid_move_penalty: int = -2
    pickup_reward: int = 20
    delivery_reward: int = 100

@dataclass
class QLearningConfig:
    alpha: float = 0.10
    gamma: float = 0.95
    epsilon: float = 1.00
    epsilon_decay: float = 0.995
    min_epsilon: float = 0.05


@dataclass
class TrainingConfig:
    episodes: int = 5000
    max_steps: int = 100

In [25]:
# ==========================
# Configuration Objects
# ==========================

In [26]:
environment = EnvironmentConfig()
rewards = RewardConfig()
q_learning = QLearningConfig()
training = TrainingConfig()

In [27]:
# ==========================
# Action Space
# ==========================

In [28]:
ACTIONS = [
    (-1, 0),      # Up
    (1, 0),       # Down
    (0, -1),      # Left
    (0, 1),       # Right
    (-1, -1),     # Up Left
    (-1, 1),      # Up Right
    (1, -1),      # Down Left
    (1, 1)        # Down Right
]

In [29]:
# ==========================
# Environment
# ==========================

In [30]:
class GridWorld:
    # Initialize the environment
    def __init__(self, config: EnvironmentConfig):
        self.size = config.grid_size
        self.goal = (
            self.size - 1,
            self.size - 1
        )

    # Start a new episode
    def reset(self):
        self.agent = self.random_position()
        self.pickup = self.random_pickup()
        self.carrying = False
        return self.get_state()

    # Generate a random grid position
    def random_position(self):
        row = np.random.randint(self.size)
        col = np.random.randint(self.size)
        return (row, col)

    # Generate a valid pickup location
    def random_pickup(self):
        pickup = self.random_position()
        while self.valid_pickup(pickup) is False:
            pickup = self.random_position()
        return pickup

    # Check whether pickup location is valid
    def valid_pickup(self, pickup):
        different_agent = pickup != self.agent
        different_goal = pickup != self.goal
        return different_agent and different_goal

    # Return current observation
    def get_state(self):
        return (
            self.agent,
            self.pickup,
            self.carrying
        )

    # Execute one action
    def step(self, action):
        reward = rewards.step_reward
        reward += self.move(action)
        reward += self.collect_reward()
        completed = self.complete_delivery()
        if completed:
            reward += rewards.delivery_reward
        return (
            self.get_state(),
            reward,
            completed
        )

    # Move the agent
    def move(self, action):
        dx, dy = ACTIONS[action]
        row = self.agent[0] + dx
        col = self.agent[1] + dy
        inside = self.inside_grid(row, col)
        if inside:
            self.agent = (row, col)
            return 0
        return rewards.invalid_move_penalty

    # Check grid boundaries
    def inside_grid(self, row, col):
        valid_row = 0 <= row < self.size
        valid_col = 0 <= col < self.size
        return valid_row and valid_col

    # Give pickup reward
    def collect_reward(self):
        pickup_cell = self.agent == self.pickup
        if pickup_cell:
            return self.pickup_item()
        return 0

    # Pick up the item
    def pickup_item(self):
        carrying_item = self.carrying
        if carrying_item:
            return 0
        self.carrying = True
        return rewards.pickup_reward

    # Check whether delivery is complete
    def complete_delivery(self):
        at_goal = self.agent == self.goal
        carrying_item = self.carrying
        return at_goal and carrying_item

    # Display the environment
    def render(self):
        grid = self.create_grid()
        self.place_pickup(grid)
        self.place_goal(grid)
        self.place_agent(grid)
        self.print_grid(grid)

    # Create an empty grid
    def create_grid(self):
        return [
            ['.' for _ in range(self.size)]
            for _ in range(self.size)
        ]

    # Place pickup location
    def place_pickup(self, grid):
        carrying_item = self.carrying
        if carrying_item:
            return
        row, col = self.pickup
        grid[row][col] = "A"

    # Place goal location
    def place_goal(self, grid):
        row, col = self.goal
        grid[row][col] = "B"

    # Place agent
    def place_agent(self, grid):
        row, col = self.agent
        if self.carrying:
            grid[row][col] = "P*"
            return
        grid[row][col] = "P"

    # Print the grid
    def print_grid(self, grid):
        print("-" * (self.size * 4))
        for row in grid:
            print(" ".join(f"{cell:>3}" for cell in row))
        print("-" * (self.size * 4))